In [0]:
import json

import data.utilities.common as Commonlib
import yaml

from data.utilities.medallion import Medallion
from data.utilities.medallion_layer import MedallionLayer as ML
from data.utilities.medallion_table_utility import MedallionTableUtility as MTU
from data.utilities.publishers import SnowflakeExternalTable

from pyspark.sql.functions import col, concat, lit, round, substring, trim, udf, when
from pyspark.sql.types import LongType

In [0]:
# # this entire command block will re-execute each time the "1. configuration file" drop-down selection is changed
config_selection_path = "./configuration/"
config_entries = Commonlib.list_files(config_selection_path, max_depth=1)
dbutils.widgets.dropdown("config_file", "", [""] + sorted(config_entries), "1. configuration file")

In [0]:
# load configs using parameter values
config_file = dbutils.widgets.get("config_file")
config_file_path = f"{config_selection_path}/{config_file}"

config = Commonlib.get_object_config(config_file_path)

In [0]:
# Instantiate medallion object
medallion = Medallion(config)

In [0]:
try:
    # Read silver.pps_ppkg table and set PDCN_CD PK column
    pps_ppkg = spark.read.table(MTU.get_fully_qualified_table_name("oracle", "upc", "pps_ppkg"))[
        [
            "generic_pdcn",
            "prod_pkg_id",
            "pkg_id",
            "product_id",
            "country_cd",
            "full_upc_carrier_cd",
            "graphic_cd",
            "full_upc_case_cd",
            "full_upc_unit_cd",
            "upc_carrier_cd",
            "upc_case_cd",
            "upc_unit_cd",
            "upc_suppress_flg",
            "status_cd",
            "__created_tsp",
        ]
    ].withColumnRenamed("prod_pkg_id", "pdcn_cd")

    # Read silver.pps_pkg table
    pps_pkg = spark.read.table(MTU.get_fully_qualified_table_name("oracle", "upc", "pps_pkg"))[
        [
            "pkg_id",
            "standard_nm",
            "cont_id",
            "carton_cd",
            "line_cd",
            "liter_fct",
            "qualifier_txt",
            "unit_package_qty",
            "price_group_cd",
            "container_unit_qty",
            "barrel_fct",
            "case_fct",
            "wrap_cd",
        ]
    ].withColumnRenamed("standard_nm", "pkg_nm")

    # Read silver.pps_prod table
    pps_prod = spark.read.table(MTU.get_fully_qualified_table_name("oracle", "upc", "pps_prod"))[
        [
            "product_id",
            "pkg_brand_cd",
            "alc_strength_cd",
            "act_alchl_strength_cd",
            "expanded_alchl_strength_cd",
        ]
    ].withColumn(
        "expanded_alchl_strength_cd", round(col("expanded_alchl_strength_cd"), 1)
    )  # Set alcohol strength value precision

    # Read silver.pps_pkg_brand table
    pps_pkg_brand = spark.read.table(MTU.get_fully_qualified_table_name("oracle", "upc", "pps_pkg_brand"))[
        [
            "pkg_brand_cd",
            "standard_nm",
            "brewed_product_id",
            "competive_nbr",
            "import_flg",
            "internal_nbr",
            "short_nm",
            "aseptic_pkg_flg",
            "type_cd",
            "trademark_nm",
        ]
    ].withColumnRenamed("standard_nm", "brand_nm")

    # Read silver.pps_pkg_grphc table
    pps_pkg_grphc = spark.read.table(MTU.get_fully_qualified_table_name("oracle", "upc", "pps_pkg_grphc"))[
        ["graphic_cd", "dsc"]
    ]

    # Read silver.vw_curr_vers_prod_pkg table
    curr_vers_prod_pkg = spark.read.table(MTU.get_fully_qualified_table_name("oracle", "upc", "vw_curr_vers_prod_pkg"))[
        ["prod_pkg_id", "gtin_carrier_id", "gtin_case_id", "gtin_unit_id"]
    ]

    pps_cont = (
        spark.read.table(MTU.get_fully_qualified_table_name("oracle", "upc", "pps_cont"))[
            [
                "cont_id",
                "neck_cd",
                "closure_cd",
                "body_cd",
                "class_cd",
                "color_cd",
                "decoration_cd",
                "material_cd",
                "return_cd",
                "size_cd",
                "type_cd",
                "aseptic_pkg_flg",
                "qualifier_txt",
                "pkg_draft_cd",
            ]
        ]
        .withColumnRenamed("type_cd", "cont_typ_cd")
        .withColumnRenamed("aseptic_pkg_flg", "cont_aseptic_flg")
        .withColumnRenamed("qualifier_txt", "cont_qlfr_txt")
    )

    mktng_cntry = spark.read.table(MTU.get_fully_qualified_table_name("oracle", "upc", "mktng_cntry"))[
        ["mktng_cntry_cd", "mktng_cntry_nm"]
    ].withColumnRenamed("mktng_cntry_cd", "country_cd")

    pps_cont_neck = spark.read.table(MTU.get_fully_qualified_table_name("oracle", "upc", "pps_cont_neck"))[
        ["neck_cd", "dsc"]
    ].withColumnRenamed("dsc", "btl_neck_typ_nm")
    pps_cont_close = spark.read.table(MTU.get_fully_qualified_table_name("oracle", "upc", "pps_cont_close"))[
        ["closure_cd", "dsc"]
    ].withColumnRenamed("dsc", "clos_dsc")
    pps_cont_body = spark.read.table(MTU.get_fully_qualified_table_name("oracle", "upc", "pps_cont_body"))[
        ["body_cd", "dsc"]
    ].withColumnRenamed("dsc", "cont_body_typ_nm")
    pps_cont_class = spark.read.table(MTU.get_fully_qualified_table_name("oracle", "upc", "pps_cont_class"))[
        ["class_cd", "dsc"]
    ].withColumnRenamed("dsc", "cont_class_nm")
    pps_cont_color = spark.read.table(MTU.get_fully_qualified_table_name("oracle", "upc", "pps_cont_color"))[
        ["color_cd", "dsc"]
    ].withColumnRenamed("dsc", "cont_colr_nm")
    pps_cont_dec = spark.read.table(MTU.get_fully_qualified_table_name("oracle", "upc", "pps_cont_dec"))[
        ["decoration_cd", "dsc"]
    ].withColumnRenamed("dsc", "cont_decor_lbl_typ_nm")
    pps_cont_mat = spark.read.table(MTU.get_fully_qualified_table_name("oracle", "upc", "pps_cont_mat"))[
        ["material_cd", "dsc"]
    ].withColumnRenamed("dsc", "cont_matl_typ_nm")
    pps_cont_size = spark.read.table(MTU.get_fully_qualified_table_name("oracle", "upc", "pps_cont_size"))[
        ["size_cd", "dsc"]
    ].withColumnRenamed("dsc", "cont_sz_nm")
    pps_cont_rcls = spark.read.table(MTU.get_fully_qualified_table_name("oracle", "upc", "pps_cont_rcls"))[
        ["return_cd", "dsc"]
    ].withColumnRenamed("dsc", "cont_rtrn_typ_nm")
    pps_cont_type = (
        spark.read.table(MTU.get_fully_qualified_table_name("oracle", "upc", "pps_cont_type"))[["type_cd", "dsc"]]
        .withColumnRenamed("dsc", "cont_typ_nm")
        .withColumnRenamed("type_cd", "cont_typ_cd")
    )
    pps_mstr_crtn = spark.read.table(MTU.get_fully_qualified_table_name("oracle", "upc", "pps_mstr_crtn"))[
        ["carton_cd", "dsc"]
    ].withColumnRenamed("dsc", "crtn_dsc")
    pps_pkg_mkt_ln = spark.read.table(MTU.get_fully_qualified_table_name("oracle", "upc", "pps_pkg_mkt_ln"))[
        ["line_cd", "dsc"]
    ].withColumnRenamed("dsc", "mkt_ln_nm")
    pps_pkg_prc_gr = spark.read.table(MTU.get_fully_qualified_table_name("oracle", "upc", "pps_pkg_prc_gr"))[
        ["price_group_cd", "dsc", "tax_cd"]
    ].withColumnRenamed("dsc", "prc_grp_nm")
    pps_pkg_wrap = spark.read.table(MTU.get_fully_qualified_table_name("oracle", "upc", "pps_pkg_wrap"))[
        ["wrap_cd", "dsc"]
    ].withColumnRenamed("dsc", "wrap_dsc")

except Exception as e:
    medallion.logger.error(e)
    print(e)
    raise

medallion.logger.info("source data loaded successfully.")

In [0]:
try:
    # Joins silver tables together
    df = pps_ppkg.join(pps_pkg, on="pkg_id", how="left")
    df = df.join(pps_prod, on="product_id", how="left")
    df = df.join(pps_pkg_brand, on="pkg_brand_cd", how="left")
    df = df.withColumn("prod_pkg_id", concat(col("product_id"), col("pkg_id")))
    df = df.join(pps_pkg_grphc, on="graphic_cd", how="left")
    df = df.join(curr_vers_prod_pkg, on="prod_pkg_id", how="left")
    df = df.join(pps_cont, on="cont_id", how="left")
    df = df.join(mktng_cntry, on="country_cd", how="left")
    df = df.join(pps_cont_neck, on="neck_cd", how="left")
    df = df.join(pps_cont_close, on="closure_cd", how="left")
    df = df.join(pps_cont_body, on="body_cd", how="left")
    df = df.join(pps_cont_class, on="class_cd", how="left")
    df = df.join(pps_cont_color, on="color_cd", how="left")
    df = df.join(pps_cont_dec, on="decoration_cd", how="left")
    df = df.join(pps_cont_mat, on="material_cd", how="left")
    df = df.join(pps_cont_size, on="size_cd", how="left")
    df = df.join(pps_cont_rcls, on="return_cd", how="left")
    df = df.join(pps_cont_type, on="cont_typ_cd", how="left")
    df = df.join(pps_mstr_crtn, on="carton_cd", how="left")
    df = df.join(pps_pkg_mkt_ln, on="line_cd", how="left")
    df = df.join(pps_pkg_prc_gr, on="price_group_cd", how="left")
    df = df.join(pps_pkg_wrap, on="wrap_cd", how="left")

    # Generate columns
    df = df.withColumn(
        "alchl_clsfn_cd",
        when(col("expanded_alchl_strength_cd") <= 4, "LOW")
        .when(col("expanded_alchl_strength_cd") >= 9, "ALT")
        .otherwise("STD"),
    )
    df = df.withColumn("brnd_prod_nm", concat(col("brand_nm"), lit(" "), col("act_alchl_strength_cd"), lit("%")))
    df = df.withColumn("genr_pdcn_cd_short", substring(col("generic_pdcn"), -5, 5))
    df = df.withColumn(
        "cont_nm",
        concat(
            col("cont_sz_nm"),
            lit(" "),
            col("cont_matl_typ_nm"),
            lit(" "),
            col("cont_typ_nm"),
        ),
    )
    df = df.withColumn(
        "pdcn_nm",
        concat(col("brand_nm"), lit(" "), col("act_alchl_strength_cd"), lit("% "), col("pkg_nm")),
    )
    df = df.withColumn("alchl_vol_pct", round(col("expanded_alchl_strength_cd") / 100, 3))
    df = df.withColumn(
        "alchl_wgt_pct", round(col("alchl_vol_pct") * 0.79336, 3)
    )  # See formula here: http://brewwiki.com/index.php/Alcohol_By_Weight

    # Apply column transformations
    clean_whitespace_columns = [
        "pdcn_nm",
        "brand_nm",
        "short_nm",
        "type_cd",
        "pkg_nm",
        "mkt_ln_nm",
        "trademark_nm",
        "full_upc_carrier_cd",
        "full_upc_case_cd",
        "full_upc_unit_cd",
        "graphic_cd",
        "brnd_prod_nm",
    ]
    remove_whitespace = udf(lambda x: x.replace("  ", " ") if x is not None else x)
    for col_name in clean_whitespace_columns:
        df = df.withColumn(col_name, remove_whitespace(col(col_name)))
        df = df.withColumn(col_name, trim(col(col_name)))

    rename_flags = udf(lambda x: "Y" if x == "A" else "N")
    df = df.withColumn("status_cd", rename_flags(col("status_cd")))

    # Rename columns and set column order
    ordered_renames = {
        "generic_pdcn": "genr_pdcn_cd",
        "genr_pdcn_cd_short": "genr_pdcn_cd_short",
        "pdcn_cd": "pdcn_cd",
        "pdcn_nm": "pdcn_nm",
        "act_alchl_strength_cd": "act_alchl_strength_cd",
        "expanded_alchl_strength_cd": "expanded_alchl_strength_cd",
        "alchl_clsfn_cd": "alchl_clsfn_cd",
        "by_volume_pct": "alchl_vol_pct",
        "by_weight_pct": "alchl_wgt_pct",
        "product_id": "brnd_prod_cd",
        "brewed_product_id": "brewd_prod_cd",
        "pkg_brand_cd": "brnd_cd",
        "aseptic_pkg_flg": "brnd_aseptic_flg",
        "competive_nbr": "brnd_cmptv_nbr",
        "import_flg": "brnd_imp_flg",
        "internal_nbr": "brnd_intrl_nbr",
        "brand_nm": "brnd_nm",
        "brnd_prod_nm": "brnd_prod_nm",
        "short_nm": "brnd_short_nm",
        "type_cd": "brnd_typ_cd",
        "pkg_id": "pkg_cd",
        "pkg_nm": "pkg_nm",
        "graphic_cd": "pkg_graphc_cd",
        "dsc": "pkg_grphc_dsc",
        "qualifier_txt": "pkg_qlfr_txt",
        "liter_fct": "pkg_ltr_equiv_fct",
        "unit_package_qty": "pkg_unit_qty",
        "price_group_cd": "prc_grp_cd",
        "prc_grp_nm": "prc_grp_nm",
        "cont_id": "cont_cd",
        "cont_nm": "cont_nm",
        "cont_qlfr_txt": "cont_qlfr_txt",
        "neck_cd": "btl_neck_typ_cd",
        "btl_neck_typ_nm": "btl_neck_typ_nm",
        "closure_cd": "clos_cd",
        "clos_dsc": "clos_dsc",
        "body_cd": "cont_body_typ_cd",
        "cont_body_typ_nm": "cont_body_typ_nm",
        "class_cd": "cont_class_cd",
        "cont_class_nm": "cont_class_nm",
        "color_cd": "cont_colr_cd",
        "cont_colr_nm": "cont_colr_nm",
        "decoration_cd": "cont_decor_lbl_typ_cd",
        "cont_decor_lbl_typ_nm": "cont_decor_lbl_typ_nm",
        "material_cd": "cont_matl_typ_cd",
        "cont_matl_typ_nm": "cont_matl_typ_nm",
        "carton_cd": "crtn_cd",
        "crtn_dsc": "crtn_dsc",
        "size_cd": "cont_sz_cd",
        "cont_sz_nm": "cont_sz_nm",
        "return_cd": "cont_rtrn_typ_cd",
        "cont_rtrn_typ_nm": "cont_rtrn_typ_nm",
        "cont_aseptic_flg": "cont_aseptic_flg",
        "line_cd": "mkt_ln_cd",
        "mkt_ln_nm": "mkt_ln_nm",
        "cont_typ_cd": "cont_typ_cd",
        "cont_typ_nm": "cont_typ_nm",
        "wrap_cd": "wrap_cd",
        "wrap_dsc": "wrap_dsc",
        "container_unit_qty": "cont_unit_qty",
        "pkg_draft_cd": "drght_flg",
        "barrel_fct": "pkg_brrl_equiv_fct",
        "case_fct": "pkg_cs_equiv_fct",
        "country_cd": "mktng_cntry_cd",
        "trademark_nm": "trdmk_nm",
        "mktng_cntry_nm": "mktng_cntry_nm",
        "full_upc_carrier_cd": "full_carr_upc_cd",
        "full_upc_case_cd": "full_case_upc_cd",
        "full_upc_unit_cd": "full_unit_upc_cd",
        "upc_carrier_cd": "carr_upc_cd",
        "upc_case_cd": "case_upc_cd",
        "upc_unit_cd": "unit_upc_cd",
        "gtin_carrier_id": "full_carr_gtin_cd",
        "gtin_case_id": "full_case_gtin_cd",
        "gtin_unit_id": "full_unit_gtin_cd",
        "status_cd": "pdcn_actv_flg",
        "__created_tsp": "__created_tsp",
        "upc_suppress_flg": "upc_suppress_flg",
    }

    for k, v in ordered_renames.items():
        df = df.withColumnRenamed(k, v)

    df = df[list(ordered_renames.values())]

except Exception as e:
    medallion.logger.error(e)
    raise

medallion.logger.info("data successfully merged and transformed.")

In [0]:
medallion.df = df

In [0]:
try:
    #     # If the target table is empty, load the historic data first.
    #     if spark.catalog.tableExists(medallion.table_lookup["gold"]):
    #         if spark.read.table(medallion.table_lookup["gold"]).count() > 0:
    #             medallion.logger.info("Table already exists in the gold layer - skipping historic load.")
    #         else:
    historic_df = spark.read.table(MTU.get_fully_qualified_table_name("oracle", "upc_bronze", "pdcn_dm_legacy"))

    # Lowercase columns
    for column in list(historic_df.columns):
        historic_df = historic_df.withColumnRenamed(column, column.lower())

    # Rename columns
    historic_df = historic_df.drop("__created_tsp")
    historic_df = historic_df.withColumnRenamed("edw_mod_tsp", "__created_tsp")

    # Subset columns
    subset_columns = [{"name": x["name"], "type": x["type"]} for x in config["gold"]["schema"]]

    for column in subset_columns:
        if column["name"] not in historic_df.columns:
            historic_df = historic_df.withColumn(column["name"], lit(None).cast(column["type"]))

    historic_df = historic_df[[c["name"] for c in subset_columns]]

    # Update ALCHL_CLSFN_CD
    historic_df = historic_df.withColumn(
        "alchl_clsfn_cd",
        when(col("act_alchl_strength_cd") <= 4, "LOW").when(col("act_alchl_strength_cd") >= 9, "ALT").otherwise("STD"),
    )
    historic_medallion = Medallion(config)
    historic_medallion.df = historic_df
    historic_medallion.write_to_delta(load_type="overwrite", layer="gold")

except Exception as e:
    medallion.logger.error(e)
    raise

In [0]:
medallion.write_to_delta(
    load_type="upsert", layer="gold", update_condition="source.pdcn_actv_flg = 'Y' and target.pdcn_actv_flg = 'Y'"
)

In [0]:
# Update EXPANDED_ALCHL_STRENGTH_CD for blank records, even if inactive
medallion.df.createOrReplaceTempView("modern_product_temp")
spark.sql(
    f"""
    merge into {medallion.table_lookup['gold']}
    using modern_product_temp
    on modern_product_temp.pdcn_cd = {medallion.table_lookup['gold']}.pdcn_cd
    when matched and {medallion.table_lookup['gold']}.expanded_alchl_strength_cd is null
    then update set {medallion.table_lookup['gold']}.expanded_alchl_strength_cd = modern_product_temp.expanded_alchl_strength_cd
    """
)

In [0]:
target_df = spark.read.table(medallion.table_lookup["gold"])  # Load all data from the target
unique_product_ids = target_df[["pdcn_cd", "pdcn_actv_flg"]].withColumn(
    "product_id", substring(col("pdcn_cd"), 0, 4)
)  # Get a unique list of product IDs
reused_product_ids = (
    unique_product_ids.select("product_id").groupBy("product_id").count().filter("count > 1")
)  # Get a list of product IDs that have been reused
reused_product_ids.createOrReplaceTempView("reused_product_ids")
removable_pdcns = unique_product_ids.filter(
    "product_id in (select product_id from reused_product_ids)"
)  # Get a list of all PDCN codes with reusable product IDs
removable_pdcns.createOrReplaceTempView("removable_pdcns")

In [0]:
spark.sql(
    f"""
    merge into {medallion.table_lookup['gold']}
    using removable_pdcns
    on removable_pdcns.pdcn_cd = {medallion.table_lookup['gold']}.pdcn_cd
    when matched and {medallion.table_lookup['gold']}.pdcn_actv_flg != 'Y'
    then delete
    """
)

In [0]:
try:
    # publish to external stage - snowflake
    if config.get("publish_to_sf", False):
        pub = SnowflakeExternalTable(config.get("catalog"), config.get("schema"), config.get("name"))
        pub.publish_ext_table()

except Exception as e:
    medallion.logger.error(e)
    raise

medallion.logger.info("publish activity completed successfully.")

In [0]:
# compare legacy and modern datasets
try:
    medallion.compare_legacy_and_modern(
        config.get("name", None),
        config["comparison_check"]["legacy_query"],
        config["comparison_check"]["modern_query"],
        medallion.key_columns,
        config.get("comparison_check", {}).get("completeness_lower_bound"),
        config.get("comparison_check", {}).get("equivalency_lower_bound"),
        config.get("comparison_check", {}).get("col_exclusions"),
        config.get("comparison_check", {}).get("auto_resolve_col_diffs"),
    )
except KeyError as e:
    medallion.logger.warning(
        "Skipping comparison of legacy and modern because the required config is missing or invalid."
    )